**Author**:
- Tianci Wang - [tiancwang@ethz.ch](tianci:tiancwang@ethz.ch)

**Date**: 06/08/2026

# Policy Portfolio Construction Code v1.0
This code is used to perform policy evaluation and build policy portfolios for each energy transition pathway based on the baseline transition pathway of EP2050+.
The basic assumption is that the current in-force policies are sufficient for the transition following the baseline scenario in the near-term period (2030–2040). With this model, we will have not only the current in-force policies but also the possible policies that may be needed in the future, and analyze all policies together to gain an understanding of the policy portfolio construction for different pathways and how the priority of each policy would change over the longer-term period (2040–2050).
## Workflow
The code consists of five main steps:
1. Data preparation:

   Import all the input data. Build a "Policy_evaluation" file for the following MCDA analysis based on the characteristics of the policies, the experts' opinions, the technologies each policy is intended to support, and the factual data of these technologies from the present to 2050.
2. MCDA conducting:

   Calculate the MCDA scores based on the "Policy_evaluation" file by choosing the TOPSIS method for both time periods (2030–2040 and 2040–2050).
3. Relevance Score calculation:

   Calculate the relevance score for each policy based on the comparison of the technology deployment levels between the EP2050+ scenario and the MIX, REMIX, and H2 scenarios, as well as the technologies each policy is intended to support.
4. Robustness analysis:

   Include both uncertainty analysis and sensitivity analysis. For the uncertainty analysis, consider uncertainties in the technology data, differences in experts' opinions, and the allocation of the policy support levels to different technologies, and use Monte Carlo simulation to run a sufficient number of iterations and observe the variation in the results. For the sensitivity analysis, only consider the sensitivity of the TOPSIS criteria weights. Analyze how much the weights can change without leading to a change in the ranking for both time periods.
5. Result visualization:

   Generate the final plots to present the relevance scores and TOPSIS scores, and generate the policy portfolio results along with the visualization of the robustness analysis results.
## Input data
1. Policy related data: Policy characteristic (Policy_data.csv), Experts' opinions (Criteria_Acceptance_Ins.csv, Criteria_Admin_Burden.csv, Criteria_Perceived_Equity.csv) ;
2. Technology related data: Technology factual data (Technology_data.csv).
3. Scenario data: The energy system results from the scenario simulation (Scenario_energy_production.csv, Scenario_installed_capacity.csv);
4. Others: Technology policy relationship (Technology_policy_matrix.csv), Experts' judgments to criteria weights (AHP_weight_input.csv, BWM_weight_input.csv).

## 1 Data Preparation
* Load all input files, verify structure and technology-name consistency.
* Build criteria data, each tested individually before combining.
  * Cost of carbon abatement: For each policy, take the technologies it supports (from Technology_policy_matrix), normalize their weights to sum to 1, then compute a weighted average of Cost_current (for 2030-2040) and Cost_2050 (for 2040-2050) from Technology_data.
  * Social Acceptance of technologies: Same logic as Cost, just a different source column. So this value is the SAME for both time periods.
  * Deployment difficulty:
    * For the EP2050+ baseline scenario, compute each technology's SHARE of
     its own sector's total production (so shares sum to 1 within a sector).
    * For each policy, take the weighted average of these shares across the
     technologies it supports (using the same normalized policy-technology
     weights as the cost or acceptance of technologies).
    * Same value for both time periods.
    * Interpretation: LOW share = technology is barely present in the baseline
     -> policy needs to work HARDER to promote it -> HIGH deployment difficulty.
     This direction (low value = harder) will be handled later by TOPSIS.
  * Cost gap: Cost_current minus Cost_2050. Only used in the SECOND TOPSIS run (2040-2050).
  * Acceptance of policy instrument, Perceived equity, Administrative burden (Qualitative expert-opinion criteria): Look up the policy's category (Instrument / Administrative_touchpoint / their combo) in a small reference table and pull the 'mode' value for main TOPSIS. Same value for BOTH time periods.
* Assemble two clean Policy_evaluation tables, ready for TOPSIS.
  * Period 1 (2030-2040): 6 criteria, using Cost_current
  * Period 2 (2040-2050): 7 criteria, using Cost_2050, PLUS Cost gap

In [2]:
# Data Import

import pandas as pd
import numpy as np

## Create file paths
DATA_DIR = "D:/Tansy/Master thesis/MCDA/data/final/input/"  # <-- change this to your folder

files = {
    "policy": "Policy_data.csv",
    "tech_policy_matrix": "Technology_policy_matrix.csv",
    "technology": "Technology_data.csv",
    "scenario_production": "Scenario_energy_production.csv",
    "scenario_capacity": "Scenario_installed_capacity.csv",
    "crit_acceptance_ins": "Criteria_Acceptance_Ins.csv",
    "crit_admin_burden": "Criteria_Admin_Burden.csv",
    "crit_perceived_equity": "Criteria_Perceived_Equity.csv",
    "AHP_input": "AHP_weight_input.csv",
    "BWM_input": "BWM_weight_input.csv"
}

## Load everything into a dictionary of DataFrames
data = {}
for key, filename in files.items():
    data[key] = pd.read_csv(DATA_DIR + filename)

## Sanity check on each table
print("=" * 70)
for key, df in data.items():
    print(f"[{key}]  ({files[key]})")
    print(f"  shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"  columns: {list(df.columns)}")
    n_missing = df.isna().sum().sum()
    print(f"  total missing values: {n_missing}")
    print("-" * 70)

[policy]  (Policy_data.csv)
  shape: 48 rows x 9 columns
  columns: ['Policy_ID', 'Status', 'Order', 'Name', 'Source_policies', 'Energy_sector', 'Promoting_technologies', 'Instrument', 'Administrative_touchpoint']
  total missing values: 0
----------------------------------------------------------------------
[tech_policy_matrix]  (Technology_policy_matrix.csv)
  shape: 48 rows x 20 columns
  columns: ['Policy_ID', 'Biomass_Boiler', 'Heat_Pump', 'Methane_Boiler', 'CHP_waste_heating', 'CHP_Methane_heating', 'CCGT_Ren_Methane', 'CHP_waste_electricity', 'CHP_Methane_electricity', 'Hydro_reservoir', 'Hydro_run_of', 'PV_Roof', 'Pumped_Hydro', 'Wind', 'CCGT_Gas_DACCS', 'Alpine_PV', 'Heavy_EV', 'Heavy_FCEV', 'Light_EV', 'Light_FCEV']
  total missing values: 768
----------------------------------------------------------------------
[technology]  (Technology_data.csv)
  shape: 19 rows x 4 columns
  columns: ['Technology', 'Acceptance', 'Cost_current', 'Cost_2050']
  total missing values: 0
----

In [3]:
# Build Policy_evaluation table

## Criterion 1: Cost of carbon abatement (EUR/tCO2eq)

### Take the Technology_policy_matrix and normalize each row so its non-missing weights sum to 1. Rows that are all-NaN stay all-NaN.
def normalize_weights(matrix_df, id_col="Policy_ID"):
    tech_cols = [c for c in matrix_df.columns if c != id_col]
    weights = matrix_df.set_index(id_col)[tech_cols]
    row_sums = weights.sum(axis=1, skipna=True)
    normalized = weights.div(row_sums, axis=0)
    return normalized  # index = Policy_ID, columns = technologies, values = normalized weights

### For each policy, compute sum( weight_i * cost_i )  over technologies i the policy supports
def compute_weighted_cost(normalized_weights, tech_df, cost_col):
    # Build a lookup: technology name -> cost value
    cost_lookup = tech_df.set_index("Technology")[cost_col]
    # Reindex so column order in normalized_weights matches cost_lookup order， any tech name mismatch becomes a NaN column。
    aligned_costs = cost_lookup.reindex(normalized_weights.columns)
    # Weighted sum
    weighted_cost = normalized_weights.mul(aligned_costs, axis=1).sum(axis=1, skipna=True)
    return weighted_cost

### Run it
weights = normalize_weights(data["tech_policy_matrix"])

### Sanity check
tech_names_in_data = set(data["technology"]["Technology"])
tech_names_in_matrix = set(weights.columns)
missing_names = tech_names_in_matrix - tech_names_in_data
if missing_names:
    print(f"WARNING: these matrix columns have NO match in Technology_data.csv: {missing_names}")
else:
    print("All technology names in the matrix match Technology_data.csv")

cost_2030_2040 = compute_weighted_cost(weights, data["technology"], "Cost_current")
cost_2040_2050 = compute_weighted_cost(weights, data["technology"], "Cost_2050")

### Print results
print("\nFirst 5 policies - Cost of carbon abatement (EUR/tCO2eq):")
preview = pd.DataFrame({
    "Cost_2030_2040": cost_2030_2040,
    "Cost_2040_2050": cost_2040_2050,
}).head()
print(preview)

### Check NaN results
n_nan = cost_2030_2040.isna().sum()
print(f"\nPolicies with missing Cost result: {n_nan} out of {len(cost_2030_2040)}")

## Criterion 2: Social Acceptance of technologies

### Run it again
acceptance_score = compute_weighted_cost(weights, data["technology"], "Acceptance")
acceptance_2030_2040 = acceptance_score
acceptance_2040_2050 = acceptance_score

print("\nFirst 5 policies - Social Acceptance of technologies (same both periods):")
preview2 = pd.DataFrame({
    "Acceptance_2030_2040": acceptance_2030_2040,
    "Acceptance_2040_2050": acceptance_2040_2050,
}).head()
print(preview2)

n_nan2 = acceptance_score.isna().sum()
print(f"\nPolicies with missing Acceptance result: {n_nan2} out of {len(acceptance_score)}")

All technology names in the matrix match Technology_data.csv

First 5 policies - Cost of carbon abatement (EUR/tCO2eq):
           Cost_2030_2040  Cost_2040_2050
Policy_ID                                
I_01          4749.681785     2950.054383
N_02          2560.649435     1347.986975
I_03          2573.741835     1648.083733
I_04          4749.681785     2950.054383
I_05          2560.649435     1347.986975

Policies with missing Cost result: 0 out of 48

First 5 policies - Social Acceptance of technologies (same both periods):
           Acceptance_2030_2040  Acceptance_2040_2050
Policy_ID                                            
I_01                      0.714                 0.714
N_02                      0.235                 0.235
I_03                      0.660                 0.660
I_04                      0.714                 0.714
I_05                      0.235                 0.235

Policies with missing Acceptance result: 0 out of 48


In [4]:
## Criterion 3: Deployment level under baseline pathway (aka deployment difficulty)

### Compute each technology's share of its sector's EP2050+ total
sector_totals = data["scenario_production"].groupby("Sector")["EP2050+"].transform("sum")
data["scenario_production"]["EP2050_share"] = data["scenario_production"]["EP2050+"] / sector_totals

print("\nSanity check - shares should sum to 1.0 within each sector:")
print(data["scenario_production"].groupby("Sector")["EP2050_share"].sum())

### Weighted average of these shares across each policy's supported technologies (reuse the compute_weighted_cost function)
deployment_level = compute_weighted_cost(weights, data["scenario_production"], "EP2050_share")
deployment_level_2030_2040 = deployment_level
deployment_level_2040_2050 = deployment_level

print("\nFirst 5 policies - Deployment level (baseline share):")
print(deployment_level.head())

n_nan3 = deployment_level.isna().sum()
print(f"\nPolicies with missing Deployment level result: {n_nan3} out of {len(deployment_level)}")

### Criterion 7: Cost gap
cost_gap = cost_2030_2040 - cost_2040_2050

print("\nFirst 5 policies - Cost gap (EUR/tCO2eq saved by 2050):")
print(cost_gap.head())


Sanity check - shares should sum to 1.0 within each sector:
Sector
Electricity    1.0
Heating        1.0
Transport      1.0
Name: EP2050_share, dtype: float64

First 5 policies - Deployment level (baseline share):
Policy_ID
I_01    0.196376
N_02    0.250000
I_03    0.033813
I_04    0.196376
I_05    0.250000
dtype: float64

Policies with missing Deployment level result: 0 out of 48

First 5 policies - Cost gap (EUR/tCO2eq saved by 2050):
Policy_ID
I_01    1799.627402
N_02    1212.662460
I_03     925.658101
I_04    1799.627402
I_05    1212.662460
dtype: float64


In [5]:
## Criteria 4-6: Acceptance of policy instrument, Perceived equity, Administrative burden (Qualitative expert-opinion criteria)

### Build the lookup function
def lookup_by_category(policy_df, category_col, ref_df, ref_category_col, value_col="mode"):
    # Standardize categories' names on BOTH sides before matching
    policy_key = policy_df[category_col].str.strip().str.lower()
    ref_key = ref_df[ref_category_col].str.strip().str.lower()

    # Build a lookup: normalized category name -> mode value
    value_lookup = pd.Series(ref_df[value_col].values, index=ref_key)

    result = policy_key.map(value_lookup)
    result.index = policy_df["Policy_ID"]
    return result

policy_df = data["policy"]

### Criterion 4: Acceptance of policy instrument
acceptance_ins = lookup_by_category(
    policy_df, "Instrument",
    data["crit_acceptance_ins"], "Instrument",
)

# Criterion 5: Administrative burden
admin_burden = lookup_by_category(
    policy_df, "Administrative_touchpoint",
    data["crit_admin_burden"], "Administrative_touchpoint",
)

# Criterion 6: Perceived equity
policy_df = policy_df.copy()
policy_df["Instrument_Administrative_touchpoint"] = (
    policy_df["Instrument"] + "_" + policy_df["Administrative_touchpoint"]
)
perceived_equity = lookup_by_category(
    policy_df, "Instrument_Administrative_touchpoint",
    data["crit_perceived_equity"], "Instrument_Administrative_touchpoint",
)

### Check for any unmatched (NaN) results
for name, series in [
    ("Acceptance of policy instrument", acceptance_ins),
    ("Administrative burden", admin_burden),
    ("Perceived equity", perceived_equity),
]:
    n_missing = series.isna().sum()
    status = "all matched" if n_missing == 0 else f"{n_missing} unmatched!"
    print(f"{name}: {status}")

print("\nFirst 5 policies - all 3 qualitative criteria:")
print(pd.DataFrame({
    "Acceptance_Ins": acceptance_ins,
    "Admin_Burden": admin_burden,
    "Perceived_Equity": perceived_equity,
}).head())

Acceptance of policy instrument: all matched
Administrative burden: all matched
Perceived equity: all matched

First 5 policies - all 3 qualitative criteria:
           Acceptance_Ins  Admin_Burden  Perceived_Equity
Policy_ID                                                
I_01                    1             3                 4
N_02                    1             3                 4
I_03                    4             3                 2
I_04                    1             3                 4
I_05                    1             3                 4


In [6]:
# Assemble Policy_evaluation tables

policy_evaluation_2030_2040 = pd.DataFrame({
    "Cost_carbon_abatement": cost_2030_2040,
    "Social_acceptance_tech": acceptance_2030_2040,
    "Deployment_difficulty": deployment_level_2030_2040,
    "Acceptance_policy_instrument": acceptance_ins,
    "Perceived_equity": perceived_equity,
    "Administrative_burden": admin_burden,
})

policy_evaluation_2040_2050 = pd.DataFrame({
    "Cost_carbon_abatement": cost_2040_2050,
    "Social_acceptance_tech": acceptance_2040_2050,
    "Deployment_difficulty": deployment_level_2040_2050,
    "Acceptance_policy_instrument": acceptance_ins,
    "Perceived_equity": perceived_equity,
    "Administrative_burden": admin_burden,
    "Cost_gap": cost_gap,  # extra column, not used in TOPSIS yet
})

print("\nPolicy_evaluation_2030_2040 (first 5 rows):")
print(policy_evaluation_2030_2040.head())
print(f"\nShape: {policy_evaluation_2030_2040.shape}")
print(f"Any missing values? {policy_evaluation_2030_2040.isna().sum().sum()}")

print("\n" + "-" * 70)
print("\nPolicy_evaluation_2040_2050 (first 5 rows):")
print(policy_evaluation_2040_2050.head())
print(f"\nShape: {policy_evaluation_2040_2050.shape}")
print(f"Any missing values? {policy_evaluation_2040_2050.isna().sum().sum()}")

# Save to CSV
policy_evaluation_2030_2040.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/Policy_evaluation_2030_2040.csv",
    index=True)  # <-- change this to your folder
policy_evaluation_2040_2050.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/policy_evaluation_2040_2050.csv",
    index=True)  # <-- change this to your folder
print("\n✅ Saved: Policy_evaluation_2030_2040.csv, Policy_evaluation_2040_2050.csv")


Policy_evaluation_2030_2040 (first 5 rows):
           Cost_carbon_abatement  Social_acceptance_tech  \
Policy_ID                                                  
I_01                 4749.681785                   0.714   
N_02                 2560.649435                   0.235   
I_03                 2573.741835                   0.660   
I_04                 4749.681785                   0.714   
I_05                 2560.649435                   0.235   

           Deployment_difficulty  Acceptance_policy_instrument  \
Policy_ID                                                        
I_01                    0.196376                             1   
N_02                    0.250000                             1   
I_03                    0.033813                             4   
I_04                    0.196376                             1   
I_05                    0.250000                             1   

           Perceived_equity  Administrative_burden  
Policy_ID         

## 2 MCDA conducting

### Decide weights of the criteria (Method: AHP or BWM)
ONE set of weights is used for both time periods (not two separate sets). Since Period 1 has 6 criteria and Period 2 has 7 (plus Cost_gap), we derive weights on the FULL 7-criteria set, then for Period 1 we drop the Cost_gap weight and rescale the remaining 6.

In the two input files for weights, the cells both scale from 1 to 9, answering "how much more important is criterion i than criterion j?"

(1=equal, 3=moderate, 5=strong, 7=very strong, 9=extreme; use 2/4/6/8 for in-between)

* AHP_weight_input.csv
  * Input: 7x7 matrix, Matrix must be reciprocal (cell [j,i] = 1 / cell [i,j]), diagonal = 1.
  * Output: weight vector (sums to 1) + Consistency Ratio (CR). CR < 0.10 is considered acceptable.
* BWM_weight_input.csv
  * Input: one row per criterion, with Is_Best/Is_Worst flags and Best_to_Criterion / Criterion_to_Worst comparison values.
  * Output: weight vector (sums to 1) + consistency indicator (xi, closer to 0 = more consistent).

### TOPSIS (both time periods)
Core idea: the best policy is the one CLOSEST to an imaginary "ideal best" combination of all criteria, and FARTHEST from an imaginary "ideal worst".
* Criteria directions:
  * Cost_carbon_abatement          - lower better (negative)
  * Social_acceptance_tech         - lower better (negative)
  * Deployment_difficulty          - lower better (negative)
  * Acceptance_policy_instrument   - higher better (positive)
  * Perceived_equity               - higher better (positive)
  * Administrative_burden          - lower better (negative)
  * Cost_gap (Period 2 only)       - lower better (negative)

In [7]:
# Criteria weights (AHP or BWM)

import scipy.optimize
from scipy.optimize import linprog

## AHP function

## Random Index table for AHP consistency check
RANDOM_INDEX = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}

### AHP weights calculation function
def ahp_weights(pairwise_matrix, criteria_names):

    n = pairwise_matrix.shape[0]
    assert pairwise_matrix.shape == (n, n)  # Matrix must be square

    # Normalize the Pairwise Comparison Matrix and Calculate Criterion Weights
    col_sums = pairwise_matrix.sum(axis=0)
    normalized = pairwise_matrix / col_sums
    weights = normalized.mean(axis=1)

    # Calculate the Consistency Ratio (CR)
    weighted_sum = pairwise_matrix @ weights
    lambda_max = (weighted_sum / weights).mean()
    CI = (lambda_max - n) / (n - 1) if n > 1 else 0.0
    RI = RANDOM_INDEX.get(n, 1.49)
    CR = CI / RI if RI > 0 else 0.0

    weights_series = pd.Series(weights, index=criteria_names, name="AHP_weight")
    return weights_series, CR

## BWM function

### BWM weights calculation function
def bwm_weights(criteria_names, best_index, worst_index, best_to_others, others_to_worst):

    n = len(criteria_names)
    n_vars = n + 1
    xi_idx = n

    A_ub, b_ub = [], []
    for j in range(n):
        a_Bj = best_to_others[j]
        row1 = [0.0] * n_vars; row1[best_index] += 1; row1[j] -= a_Bj; row1[xi_idx] = -1
        A_ub.append(row1); b_ub.append(0.0)
        row2 = [0.0] * n_vars; row2[best_index] -= 1; row2[j] += a_Bj; row2[xi_idx] = -1
        A_ub.append(row2); b_ub.append(0.0)

        a_jW = others_to_worst[j]
        row3 = [0.0] * n_vars; row3[j] += 1; row3[worst_index] -= a_jW; row3[xi_idx] = -1
        A_ub.append(row3); b_ub.append(0.0)
        row4 = [0.0] * n_vars; row4[j] -= 1; row4[worst_index] += a_jW; row4[xi_idx] = -1
        A_ub.append(row4); b_ub.append(0.0)

    A_eq = [[1.0] * n + [0.0]]
    b_eq = [1.0]
    bounds = [(0, 1)] * n + [(0, None)]
    c = [0.0] * n + [1.0]

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not result.success:
        raise RuntimeError(f"BWM optimization failed: {result.message}")

    weights = result.x[:n]
    xi = result.x[xi_idx]
    weights_series = pd.Series(weights, index=criteria_names, name="BWM_weight")
    return weights_series, xi

### Call the BWM function
def bwm_weights_from_csv(df):

    criteria_names = df["Criterion"].tolist()
    best_index = df.index[df["Is_Best"] == 1][0]
    worst_index = df.index[df["Is_Worst"] == 1][0]
    best_to_others = df["Best_to_Criterion"].tolist()
    others_to_worst = df["Criterion_to_Worst"].tolist()
    return bwm_weights(criteria_names, best_index, worst_index, best_to_others, others_to_worst)

## Create a Subset from All Criteria and Renormalize Criterion Weights
def subset_and_renormalize(weights_series, subset_criteria):
    subset = weights_series.loc[subset_criteria]
    return subset / subset.sum()


## Load the criteria set from both weight input files
criteria_p2 = ["Cost_carbon_abatement", "Social_acceptance_tech", "Deployment_difficulty",
               "Acceptance_policy_instrument", "Perceived_equity", "Administrative_burden",
               "Cost_gap"]
criteria_p1 = criteria_p2[:-1]  # everything except Cost_gap

## Weight calculation

### AHP 2040-2050
print(f"\n--- AHP: {len(criteria_p2)}-criteria weights")
ahp_full_weights, ahp_cr = ahp_weights(data["AHP_input"].iloc[:,1:].values, data["AHP_input"].columns[1:].tolist())
print(ahp_full_weights)
print(f"Consistency Ratio: {ahp_cr:.4f}  (should be < 0.10 to be considered acceptable)")

### BWM 2040-2050
print(f"\n--- BWM: {len(criteria_p2)}-criteria weights")
bwm_full_weights, bwm_xi = bwm_weights_from_csv(data["BWM_input"])
print(bwm_full_weights)
print(f"Consistency indicator (xi): {bwm_xi:.4f}  (closer to 0 = more consistent)")

## Choose which method feeds TOPSIS, AHP by default

weights_for_topsis = ahp_full_weights   # swap this one line to bwm_full_weights to use BWM instead

## Derive the Period 1 (6-criteria) weight vector
weights_p1 = subset_and_renormalize(weights_for_topsis, criteria_p1)
weights_p2 = weights_for_topsis  # already matches all 7 criteria

print(f"\n--- Final weights used in TOPSIS ---")
print("Period 1 (6 criteria):")
print(weights_p1)
print(f"Sum: {weights_p1.sum():.4f}  (should be 1.0)")
print("\nPeriod 2 (7 criteria):")
print(weights_p2)
print(f"Sum: {weights_p2.sum():.4f}  (should be 1.0)")


--- AHP: 7-criteria weights
Cost_carbon_abatement           0.142857
Social_acceptance_tech          0.142857
Deployment_difficulty           0.142857
Acceptance_policy_instrument    0.142857
Perceived_equity                0.142857
Administrative_burden           0.142857
Cost_gap                        0.142857
Name: AHP_weight, dtype: float64
Consistency Ratio: 0.0000  (should be < 0.10 to be considered acceptable)

--- BWM: 7-criteria weights
Cost_carbon_abatement           0.142857
Social_acceptance_tech          0.142857
Deployment_difficulty           0.142857
Acceptance_policy_instrument    0.142857
Perceived_equity                0.142857
Administrative_burden           0.142857
Cost_gap                        0.142857
Name: BWM_weight, dtype: float64
Consistency indicator (xi): 0.0000  (closer to 0 = more consistent)

--- Final weights used in TOPSIS ---
Period 1 (6 criteria):
Cost_carbon_abatement           0.166667
Social_acceptance_tech          0.166667
Deployment_diffic

In [8]:
# TOPSIS

## Build TOPSIS function

def topsis(decision_df, weights, directions):

    cols = decision_df.columns.tolist()
    w = np.array([weights[c] for c in cols])
    dirs = [directions[c] for c in cols]

    X = decision_df.values.astype(float)

    # Step 1: vector normalization (each column divided by its Euclidean norm)
    norms = np.sqrt((X ** 2).sum(axis=0))
    X_norm = X / norms

    # Step 2: apply weights
    X_weighted = X_norm * w

    # Step 3: ideal best / ideal worst (nadir) per column, by direction
    ideal_best = np.where(
        np.array(dirs) == "positive",
        X_weighted.max(axis=0),
        X_weighted.min(axis=0),
    )
    ideal_worst = np.where(
        np.array(dirs) == "positive",
        X_weighted.min(axis=0),
        X_weighted.max(axis=0),
    )

    # Step 4: Euclidean distances
    dist_to_ideal = np.sqrt(((X_weighted - ideal_best) ** 2).sum(axis=1))
    dist_to_nadir = np.sqrt(((X_weighted - ideal_worst) ** 2).sum(axis=1))

    # Step 5: TOPSIS Score (closeness coefficient)
    topsis_score = dist_to_nadir / (dist_to_ideal + dist_to_nadir)

    result = pd.DataFrame({
        "TOPSIS Score": topsis_score,
        "Distance to Ideal": dist_to_ideal,
        "Distance to Nadir": dist_to_nadir,
    }, index=decision_df.index)

    # Add one Score column per criterion
    for i, c in enumerate(cols):
        result[f"{c} Score"] = X_norm[:, i]

    # Step 6: rank (1 = best).
    result["Rank"] = result["TOPSIS Score"].rank(ascending=False).astype(int)

    # Final column order, as requested
    score_cols = [f"{c} Score" for c in cols]
    result = result[["Rank", "TOPSIS Score", "Distance to Ideal", "Distance to Nadir"] + score_cols]
    return result


## Criteria directions
directions_p1 = {
    "Cost_carbon_abatement": "negative",
    "Social_acceptance_tech": "negative",
    "Deployment_difficulty": "negative",
    "Acceptance_policy_instrument": "positive",
    "Perceived_equity": "positive",
    "Administrative_burden": "negative",
}
directions_p2 = {**directions_p1, "Cost_gap": "negative"}

## Run TOPSIS
### Period 1 (2030-2040): 6 criteria
topsis_p1 = topsis(policy_evaluation_2030_2040, weights_p1, directions_p1)

print("\n--- TOPSIS results: Period 1 (2030-2040) (first 5 rows) ---")
print(topsis_p1.head())
print(f"\nTotal policies ranked: {len(topsis_p1)} (should be 48)")

### Period 2 (2040-2050): 7 criteria (incl. Cost_gap)
topsis_p2 = topsis(policy_evaluation_2040_2050, weights_p2, directions_p2)

print("\n--- TOPSIS results: Period 2 (2040-2050) (first 5 rows) ---")
print(topsis_p2.head())
print(f"\nTotal policies ranked: {len(topsis_p2)} (should be 48)")

# ---- Save results -------------------------------------------------------------
topsis_p1.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2030_2040.csv", index=True)  # <-- change this to your folder
topsis_p2.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2040_2050.csv", index=True)  # <-- change this to your folder
print("\n Saved: TOPSIS_results_2030_2040.csv, TOPSIS_results_2040_2050.csv")


--- TOPSIS results: Period 1 (2030-2040) (first 5 rows) ---
           Rank  TOPSIS Score  Distance to Ideal  Distance to Nadir  \
Policy_ID                                                             
I_01         35      0.622069           0.051664           0.085038   
N_02         27      0.708206           0.038598           0.093681   
I_03         21      0.732089           0.037559           0.102633   
I_04         35      0.622069           0.051664           0.085038   
I_05         27      0.708206           0.038598           0.093681   

           Cost_carbon_abatement Score  Social_acceptance_tech Score  \
Policy_ID                                                              
I_01                          0.184879                      0.179037   
N_02                          0.099672                      0.058927   
I_03                          0.100182                      0.165496   
I_04                          0.184879                      0.179037   
I_05     

## 3 Relevance Score calculation
### Relevance Score
For each of the 3 alternative scenarios (MIX, REMIX, H2), measure how much each technology needs to increase/decrease relative to the EP2050+ baseline, then use the technology-policy matrix to turn that into a per-policy relevance score for each scenario.

* Formula per technology: relevance_tech = (Scenario_value - EP2050+_value) / EP2050+_sector_total
  * Positive = scenario needs MORE of this technology than the baseline
   * Negative = scenario needs LESS
   * ~0 = no change of this technology under this scenario

* Per-policy score: Relevance Score = weighted average of relevance_tech across the policy's supported technologies
 (same normalized Technology_policy_matrix weights used throughout this pipeline).

### Classify into 5 groups
To classify each policy's relevance score (per scenario) into 5 groups, use Ckmeans.1d.dp (Wang & Song, 2011), the exact optimal 1D clustering method, which finds group boundaries that minimize within-group variance.

* Groups: Strong negative | Weak negative | Neutral | Weak positive | Strong positive

Make sure the relevance score 0 always falls inside Neutral group, since this score implicates that the policy doesn't need to be adjusted in the alternative scenario.


In [16]:
# Relevance score

## technology-level relevance (increase/decrease vs EP2050+)
sector_totals_ep2050 = data["scenario_production"].groupby("Sector")["EP2050+"].transform("sum")

alt_scenarios = ["MIX", "REMIX", "H2"]
for scen in alt_scenarios:
    data["scenario_production"][f"relevance_{scen}"] = (
        (data["scenario_production"][scen] - data["scenario_production"]["EP2050+"]) / sector_totals_ep2050
    )

### Save and print intermediate results
relevance_tech = data["scenario_production"][["Sector", "Technology"] + [f"relevance_{s}" for s in alt_scenarios]]
relevance_tech.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/Technology_relevance.csv", index=False)  # <-- change this to your folder
print("\nTechnology-level relevance (first 5 rows):")
print(relevance_tech.head())
print("\n Saved: Technology_relevance.csv")

## Weight average across each policy's supported technologies

tech_relevance_lookup = data["scenario_production"].set_index("Technology")[
    [f"relevance_{s}" for s in alt_scenarios]
].reset_index()

policy_relevance = pd.DataFrame(index=weights.index)
for scen in alt_scenarios:
    policy_relevance[f"Relevance_{scen}"] = compute_weighted_cost(
        weights, tech_relevance_lookup, f"relevance_{scen}"
    )

print("\nPolicy-level relevance scores (first 5 policies):")
print(policy_relevance.head())

n_missing = policy_relevance.isna().sum().sum()
print(f"\nAny missing values? {n_missing}")


## Classify into 5 groups by Ckmeans.1d.dp

### Build Ckmeans.1d.dp fucntion
def ckmeans_1d_dp(values, n_classes):

    data = sorted(values)
    n = len(data)

    lower_class_limits = [[0] * (n_classes + 1) for _ in range(n + 1)]
    variance_combinations = [[float("inf")] * (n_classes + 1) for _ in range(n + 1)]
    variance_combinations[1][1] = 0.0

    for i in range(2, n + 1):
        sum_, sum_squares, w = 0.0, 0.0, 0.0
        for m in range(1, i + 1):
            lower_class_limit = i - m + 1
            val = data[lower_class_limit - 1]
            sum_ += val
            sum_squares += val * val
            w += 1
            variance = sum_squares - (sum_ * sum_) / w
            i4 = lower_class_limit - 1
            if i4 != 0:
                for j in range(2, n_classes + 1):
                    candidate = variance + variance_combinations[i4][j - 1]
                    if variance_combinations[i][j] >= candidate:
                        lower_class_limits[i][j] = lower_class_limit
                        variance_combinations[i][j] = candidate
        lower_class_limits[i][1] = 1
        variance_combinations[i][1] = sum_squares - (sum_ * sum_) / w

    # Backtrack through the DP table to recover the actual breakpoint values
    k = n
    kclass = [0] * (n_classes + 1)
    kclass[n_classes] = data[-1]
    kclass[0] = data[0]
    count_num = n_classes
    while count_num >= 2:
        idx = int(lower_class_limits[k][count_num] - 2)
        kclass[count_num - 1] = data[idx]
        k = int(lower_class_limits[k][count_num] - 1)
        count_num -= 1

    return kclass

### Function to make sure 0 is inside the "Neutral" group
def enforce_zero_in_neutral(breaks, neutral_index):

    adjusted = list(breaks)
    lower_idx, upper_idx = neutral_index, neutral_index + 1
    if adjusted[lower_idx] > 0:
        adjusted[lower_idx] = 0.0
    if adjusted[upper_idx] < 0:
        adjusted[upper_idx] = 0.0
    return adjusted

### Build labeling function for each group after classification
def ckmeans_classify(values, n_classes, labels, guarantee_zero_neutral=True):

    breaks = ckmeans_1d_dp(values.tolist(), n_classes)

    if guarantee_zero_neutral and "Neutral" in labels:
        neutral_index = labels.index("Neutral")
        breaks = enforce_zero_in_neutral(breaks, neutral_index)

        lower_bound = breaks[neutral_index]
        upper_bound = breaks[neutral_index + 1]

        def assign(v):
            if lower_bound <= v <= upper_bound:
                return labels[neutral_index]
            if v < lower_bound:
                for i in range(neutral_index):
                    if v <= breaks[i + 1]:
                        return labels[i]
                return labels[neutral_index - 1]
            else:
                for i in range(neutral_index + 1, n_classes):
                    if v <= breaks[i + 1]:
                        return labels[i]
                return labels[-1]
    else:
        def assign(v):
            for i in range(n_classes):
                if v <= breaks[i + 1]:
                    return labels[i]
            return labels[-1]

    return values.apply(assign), breaks


### Apply Ckmeans.1d.dp classification to each scenario's relevance score
group_labels = ["Strong negative", "Weak negative", "Neutral", "Weak positive", "Strong positive"]

for scen in alt_scenarios:
    col = f"Relevance_{scen}"
    group_col = f"Group_{scen}"
    labels_series, breaks = ckmeans_classify(policy_relevance[col], 5, group_labels)
    policy_relevance[group_col] = labels_series
    print(f"\n--- Ckmeans.1d.dp breaks for {scen} ---")
    print("Breakpoints:", [round(b, 4) for b in breaks])
    print(policy_relevance[group_col].value_counts().reindex(group_labels))

print("\nFull relevance table (first 10 policies):")
print(policy_relevance.head(10))

policy_relevance.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Policy_relevance_scores.csv", index=True)  # <-- change this to your folder
print("\n Saved: Policy_relevance_scores.csv")


Technology-level relevance (first 5 rows):
    Sector           Technology  relevance_MIX  relevance_REMIX  relevance_H2
0  Heating       Biomass_Boiler      -0.457983        -0.203081      0.030812
1  Heating            Heat_Pump       0.537815         0.236695     -0.257703
2  Heating       Methane_Boiler       0.000000         0.000000      0.226891
3  Heating    CHP_waste_heating       0.000000         0.000000      0.000000
4  Heating  CHP_Methane_heating      -0.036415        -0.036415      0.000000

 Saved: Technology_relevance.csv

Policy-level relevance scores (first 5 policies):
           Relevance_MIX  Relevance_REMIX  Relevance_H2
Policy_ID                                              
I_01            0.009204        -0.000040 -6.938894e-18
N_02            0.000000         0.000000  1.734723e-17
I_03           -0.033813        -0.033813  0.000000e+00
I_04            0.009204        -0.000040 -6.938894e-18
I_05            0.000000         0.000000  1.734723e-17

Any missin

## 4 Robustness analysis
### Weight Sensitivity Analysis
For each criterion, find how far its weight can move (up and down), while proportionally rescaling all other criteria weights to preserve their relative ratio to each other, before the final rank order of all 48 policies changes.

To achieve this, do a fine-grained scan of possible weights (from 0 to 1) for each criterion, rescaling the rest each time, re-running TOPSIS, and comparing the resulting Rank column to the baseline. Report the widest contiguous range around the original weight where ranks stay identical.
### Uncertainty Analysis
1. Compensatory relevance score

   Add the Relevance score (per alternative scenario: MIX, REMIX, H2) as an EXTRA criterion in TOPSIS, weighted at 0.5 with all original criteria weights rescaled down proportionally so everything still sums to 1.
   * Direction: positive
   * Produce 6 TOPSIS runs total: 2 time periods x 3 alternative scenarios.
2. Monte Carlo Uncertainty Analysis

   There are four perturbed inputs in the Monte Carlo analysis. Each iteration randomly perturbs 4 uncertain inputs, rebuilds the entire Policy_evaluation tables + relevance scores from scratch using those perturbed inputs, then re-runs TOPSIS (both periods) and Ckmeans (all 3 scenarios). (In TOPSIS, weights stay FIXED across iterations, only the underlying CRITERIA VALUES vary.) Finally, save Rank + TOPSIS Score, and Relevance + Group.

   The 4 perturbed inputs are:

   * Technology_policy_matrix (per policy, per supported technology):
     * Mature technologies (Biomass_Boiler, Methane_Boiler, CHP_Methane_heating, CCGT_Ren_Methane, CHP_Methane_electricity): uniform[current, current*2]
     * PV_Roof, ONLY in policies supporting EXACTLY {PV_Roof, Alpine_PV}: uniform[current, current*2]
     * Light_EV, in ANY policy where it appears alongside other technologies: uniform[current, ccurrent*2]
     * CHP_Methane_electricity: NOT sampled independently - always forced to CHP_Methane_heating_value * (0.44/0.46), even if that falls outside its own +0.5 window
     * Everything else: uniform[0, current]
     * Policies with only 1 supported technology: skipped (a single value always normalizes to 1.0)
     * After perturbing, each policy's row is renormalized to sum to 1
   * Cost_current and Cost_2050 (per technology)
      * each independently sampled from a UNIFORM distribution in [0.8x, 1.2x] of the original value
   * Cost_gap
      * recomputed as (perturbed Cost_current - perturbed Cost_2050), weighted-averaged using this iteration's perturbed technology-policy matrix
   * Qualitative criteria (Acceptance_Ins, Admin_Burden, Perceived_Equity)
      * each sampled from a DISCRETE UNIFORM distribution over the integers in [low, high] (every whole number equally likely)

   NOT perturbed
   * Social_acceptance_tech and Deployment_difficulty use their ORIGINAL technology-level values
   * Recompute each iteration using the iteration's perturbed matrix, since the technology-policy matrix affects every matrix-weighted criterion, not just cost.

In [17]:
# Weight Sensitivity Analysis

## Rescale function for the non-targeting criteria
def rescale_others(weights_series, target_criterion, target_weight):

    others = weights_series.drop(target_criterion)
    original_others_sum = others.sum()
    scale_factor = (1 - target_weight) / original_others_sum
    new_weights = others * scale_factor
    new_weights[target_criterion] = target_weight
    return new_weights.reindex(weights_series.index)

## Function for finding the weight sensitivity range
def find_sensitivity_range(decision_df, base_weights, directions, target_criterion, baseline_ranks, step=0.005):

    current_w = base_weights[target_criterion]

    # Scan upward
    max_w = current_w
    w = current_w
    while w + step <= 1.0:
        w += step
        trial_weights = rescale_others(base_weights, target_criterion, w)
        trial_result = topsis(decision_df, trial_weights, directions)
        if not trial_result["Rank"].equals(baseline_ranks):
            break
        max_w = w

    # Scan downward
    min_w = current_w
    w = current_w
    while w - step >= 0.0:
        w -= step
        trial_weights = rescale_others(base_weights, target_criterion, w)
        trial_result = topsis(decision_df, trial_weights, directions)
        if not trial_result["Rank"].equals(baseline_ranks):
            break
        min_w = w

    return min_w, max_w

## Function to calculate the range for each criterion's weight
def run_sensitivity_analysis(decision_df, base_weights, directions, period_label):
    baseline_result = topsis(decision_df, base_weights, directions)
    baseline_ranks = baseline_result["Rank"]

    rows = []
    for criterion in base_weights.index:
        min_w, max_w = find_sensitivity_range(
            decision_df, base_weights, directions, criterion, baseline_ranks
        )
        rows.append({
            "Period": period_label,
            "Criterion": criterion,
            "Current_Weight": base_weights[criterion],
            "Min_Weight_No_Rank_Change": min_w,
            "Max_Weight_No_Rank_Change": max_w,
            "Allowed_Decrease": base_weights[criterion] - min_w,
            "Allowed_Increase": max_w - base_weights[criterion],
        })
        print(f"  [{period_label}] {criterion}: current={base_weights[criterion]:.4f}, "
              f"range=[{min_w:.4f}, {max_w:.4f}]")
    return pd.DataFrame(rows)

## Print and save the sensitivity analysis results
print("\n--- Period 1 (2030-2040) sensitivity ---")
sens_p1 = run_sensitivity_analysis(policy_evaluation_2030_2040, weights_p1, directions_p1, "2030_2040")

print("\n--- Period 2 (2040-2050) sensitivity ---")
sens_p2 = run_sensitivity_analysis(policy_evaluation_2040_2050, weights_p2, directions_p2, "2040_2050")

sensitivity_results = pd.concat([sens_p1, sens_p2], ignore_index=True)
sensitivity_results.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Weight_sensitivity_results.csv", index=False)  # <-- change this to your folder
print("\n Saved: Weight_sensitivity_results.csv")
print(sensitivity_results)


--- Period 1 (2030-2040) sensitivity ---
  [2030_2040] Cost_carbon_abatement: current=0.1667, range=[0.1667, 0.1667]
  [2030_2040] Social_acceptance_tech: current=0.1667, range=[0.1667, 0.1667]
  [2030_2040] Deployment_difficulty: current=0.1667, range=[0.1667, 0.1667]
  [2030_2040] Acceptance_policy_instrument: current=0.1667, range=[0.1617, 0.1667]
  [2030_2040] Perceived_equity: current=0.1667, range=[0.1667, 0.1667]
  [2030_2040] Administrative_burden: current=0.1667, range=[0.1667, 0.1667]

--- Period 2 (2040-2050) sensitivity ---
  [2040_2050] Cost_carbon_abatement: current=0.1429, range=[0.1429, 0.1429]
  [2040_2050] Social_acceptance_tech: current=0.1429, range=[0.1429, 0.1429]
  [2040_2050] Deployment_difficulty: current=0.1429, range=[0.1429, 0.1429]
  [2040_2050] Acceptance_policy_instrument: current=0.1429, range=[0.1429, 0.1429]
  [2040_2050] Perceived_equity: current=0.1429, range=[0.1429, 0.1429]
  [2040_2050] Administrative_burden: current=0.1429, range=[0.1429, 0.1429

In [15]:
# Compensatory relevance score

relevance_df = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Policy_relevance_scores.csv", index_col=0)    # <-- change this to your folder
alt_scenarios = ["MIX", "REMIX", "H2"]

## Add weight for relevance score and rescale
def add_relevance_weight(base_weights, relevance_weight=0.5):

    rescaled = base_weights * (1 - relevance_weight)
    rescaled["Relevance"] = relevance_weight
    return rescaled


results_with_relevance = {}

## Run TOPSIS with one more criterion - relevance score
for period_label, decision_df, base_weights, base_directions in [
    ("2030_2040", policy_evaluation_2030_2040, weights_p1, directions_p1),
    ("2040_2050", policy_evaluation_2040_2050, weights_p2, directions_p2),
]:
    for scen in alt_scenarios:
        combined_df = decision_df.copy()
        combined_df["Relevance"] = relevance_df[f"Relevance_{scen}"]

        combined_weights = add_relevance_weight(base_weights, relevance_weight=0.5)
        combined_directions = {**base_directions, "Relevance": "positive"}

        result = topsis(combined_df, combined_weights, combined_directions)
        key = f"{period_label}_{scen}"
        results_with_relevance[key] = result

        fname = f"TOPSIS_with_relevance_{key}.csv"
        result.to_csv(rf"D:/Tansy/Master thesis/MCDA/data/final/output/{fname}", index=True)    # <-- change this to your folder
        print(f"\n--- {key} (weights: {dict(combined_weights.round(4))}) ---")
        print(result.head())
        print(f"Saved: {fname}")

print(f"\n All 6 TOPSIS-with-relevance runs complete and saved.")


--- 2030_2040_MIX (weights: {'Cost_carbon_abatement': np.float64(0.0833), 'Social_acceptance_tech': np.float64(0.0833), 'Deployment_difficulty': np.float64(0.0833), 'Acceptance_policy_instrument': np.float64(0.0833), 'Perceived_equity': np.float64(0.0833), 'Administrative_burden': np.float64(0.0833), 'Relevance': np.float64(0.5)}) ---
           Rank  TOPSIS Score  Distance to Ideal  Distance to Nadir  \
Policy_ID                                                             
I_01         13      0.470791           0.353334           0.314330   
N_02         32      0.462447           0.359043           0.308877   
I_03         46      0.429629           0.381527           0.287383   
I_04         13      0.470791           0.353334           0.314330   
I_05         32      0.462447           0.359043           0.308877   

           Cost_carbon_abatement Score  Social_acceptance_tech Score  \
Policy_ID                                                              
I_01                

In [22]:
# Monte Carlo Uncertainty Analysis

## Set N_ITERATIONS - can be changed for a bigger/smaller run
N_ITERATIONS = 1000
RANDOM_SEED = 42  # for reproducibility - re-run with a different seed to sanity-check stability
np.random.seed(RANDOM_SEED)

## Monte Carlo draw of the Technology_policy_matrix

MATURE_TECHS = ["Biomass_Boiler", "Methane_Boiler", "CHP_Methane_heating",
                 "CCGT_Ren_Methane", "CHP_Methane_electricity"]
PV_EXCEPTION_PAIR = {"PV_Roof", "Alpine_PV"}

raw_tpm = data["tech_policy_matrix"].set_index("Policy_ID")
tech_cols_all = raw_tpm.columns.tolist()
tech_data_raw = data["technology"].set_index("Technology")

### Pre-identify which policies match the PV_Roof exception (support EXACTLY {PV_Roof, Alpine_PV}, nothing else)
pv_exception_policies = set()
for pid, row in raw_tpm.iterrows():
    supported = set(c for c in tech_cols_all if pd.notna(row[c]))
    if supported == PV_EXCEPTION_PAIR:
        pv_exception_policies.add(pid)
print(f"PV_Roof exception applies to policies: {sorted(pv_exception_policies)}")

### Function to perturb Technology_policy_matrix
def perturb_matrix_once():

    perturbed = raw_tpm.copy()

    for pid, row in raw_tpm.iterrows():
        supported = [c for c in tech_cols_all if pd.notna(row[c])]
        if len(supported) <= 1:
            continue  # single-tech policy: perturbation is a no-op after normalization

        for tech in supported:
            if tech == "CHP_Methane_electricity":
                continue  # handled below, via the fixed ratio to heating
            current_val = row[tech]
            if tech in MATURE_TECHS:
                new_val = np.random.uniform(current_val, current_val * 2)
            elif tech == "PV_Roof" and pid in pv_exception_policies:
                new_val = np.random.uniform(current_val, current_val * 2)
            elif tech == "Light_EV":
                new_val = np.random.uniform(current_val, current_val * 2)
            else:
                new_val = np.random.uniform(0.0, current_val)
            perturbed.loc[pid, tech] = new_val

        # CHP_Methane_electricity always follows the 46:44 ratio to heating
        if "CHP_Methane_electricity" in supported:
            if "CHP_Methane_heating" in supported:
                perturbed.loc[pid, "CHP_Methane_electricity"] = (
                    perturbed.loc[pid, "CHP_Methane_heating"] * (0.44 / 0.46)
                )
            else:
                current_val = row["CHP_Methane_electricity"]
                perturbed.loc[pid, "CHP_Methane_electricity"] = np.random.uniform(
                    current_val, current_val * 2
                )

    row_sums = perturbed[tech_cols_all].sum(axis=1, skipna=True)
    normalized = perturbed[tech_cols_all].div(row_sums, axis=0)
    return normalized

## Monte Carlo draw of Cost_current and Cost_2050
def perturb_costs_once():

    n = len(tech_data_raw)
    factor_current = np.random.uniform(0.8, 1.2, size=n)
    factor_2050 = np.random.uniform(0.8, 1.2, size=n)
    out = pd.DataFrame({
        "Technology": tech_data_raw.index,
        "Cost_current": tech_data_raw["Cost_current"].values * factor_current,
        "Cost_2050": tech_data_raw["Cost_2050"].values * factor_2050,
    })
    return out

## Monte Carlo draw of the 3 qualitative criteria tables
def perturb_qualitative_once():

    def sample_df(df):
        d = df.copy()
        d["sampled"] = [np.random.randint(lo, hi + 1) for lo, hi in zip(d["low"], d["high"])]
        return d
    return (
        sample_df(data["crit_acceptance_ins"]),
        sample_df(data["crit_admin_burden"]),
        sample_df(data["crit_perceived_equity"]),
    )

## Function for one run with the single Monte Carlo draw
def run_one_iteration():

    mc_weights = perturb_matrix_once()
    mc_costs = perturb_costs_once()
    mc_acc_ins, mc_admin, mc_equity = perturb_qualitative_once()

    mc_cost_2030_2040 = compute_weighted_cost(mc_weights, mc_costs, "Cost_current")
    mc_cost_2040_2050 = compute_weighted_cost(mc_weights, mc_costs, "Cost_2050")
    mc_cost_gap = mc_cost_2030_2040 - mc_cost_2040_2050

    mc_acceptance = compute_weighted_cost(mc_weights, data["technology"], "Acceptance")
    mc_deployment = compute_weighted_cost(mc_weights, data["scenario_production"], "EP2050_share")

    mc_acceptance_ins = lookup_by_category(data["policy"], "Instrument", mc_acc_ins, "Instrument", value_col="sampled")
    mc_admin_burden = lookup_by_category(data["policy"], "Administrative_touchpoint", mc_admin, "Administrative_touchpoint", value_col="sampled")
    policy_df_combo = data["policy"].copy()
    policy_df_combo["Instrument_Administrative_touchpoint"] = (
        policy_df_combo["Instrument"] + "_" + policy_df_combo["Administrative_touchpoint"]
    )
    mc_perceived_equity = lookup_by_category(policy_df_combo, "Instrument_Administrative_touchpoint",
                                              mc_equity, "Instrument_Administrative_touchpoint", value_col="sampled")

    mc_eval_p1 = pd.DataFrame({
        "Cost_carbon_abatement": mc_cost_2030_2040,
        "Social_acceptance_tech": mc_acceptance,
        "Deployment_difficulty": mc_deployment,
        "Acceptance_policy_instrument": mc_acceptance_ins,
        "Perceived_equity": mc_perceived_equity,
        "Administrative_burden": mc_admin_burden,
    })
    mc_eval_p2 = pd.DataFrame({
        "Cost_carbon_abatement": mc_cost_2040_2050,
        "Social_acceptance_tech": mc_acceptance,
        "Deployment_difficulty": mc_deployment,
        "Acceptance_policy_instrument": mc_acceptance_ins,
        "Perceived_equity": mc_perceived_equity,
        "Administrative_burden": mc_admin_burden,
        "Cost_gap": mc_cost_gap,
    })

    mc_topsis_p1 = topsis(mc_eval_p1, weights_p1, directions_p1)
    mc_topsis_p2 = topsis(mc_eval_p2, weights_p2, directions_p2)

    mc_relevance = pd.DataFrame(index=mc_weights.index)
    for scen in alt_scenarios:
        mc_relevance[f"Relevance_{scen}"] = compute_weighted_cost(
            mc_weights, tech_relevance_lookup, f"relevance_{scen}"
        )
    mc_relevance_groups = pd.DataFrame(index=mc_weights.index)
    for scen in alt_scenarios:
        labels_series, _ = ckmeans_classify(mc_relevance[f"Relevance_{scen}"], 5, group_labels)
        mc_relevance_groups[f"Group_{scen}"] = labels_series

    return mc_topsis_p1, mc_topsis_p2, mc_relevance, mc_relevance_groups


## Test run with 9 iterations first
print("\n--- test run: 3 quick iterations to check for errors before scaling up ---")
for i in range(3):
    p1, p2, rel, grp = run_one_iteration()
    print(f"Iteration {i}: Period1 top policy = {p1.index[p1["Rank"] == 1][0]}, "
          f"Period2 top policy = {p2.index[p1["Rank"] == 1][0]}, "
          f"Relevance_MIX range = [{rel['Relevance_MIX'].min():.3f}, {rel['Relevance_MIX'].max():.3f}]")
print(" Smoke test passed, no errors")

## Full Monte Carlo run
import time
print(f"\n--- Running full Monte Carlo: {N_ITERATIONS} iterations ---")
start_time = time.time()

topsis_p1_rows = []
topsis_p2_rows = []
relevance_rows = []

for it in range(N_ITERATIONS):
    p1, p2, rel, grp = run_one_iteration()

    p1_copy = p1[["Rank", "TOPSIS Score"]].copy()
    p1_copy["Iteration"] = it
    p1_copy["Policy_ID"] = p1_copy.index
    topsis_p1_rows.append(p1_copy)

    p2_copy = p2[["Rank", "TOPSIS Score"]].copy()
    p2_copy["Iteration"] = it
    p2_copy["Policy_ID"] = p2_copy.index
    topsis_p2_rows.append(p2_copy)

    for scen in alt_scenarios:
        rel_copy = pd.DataFrame({
            "Iteration": it,
            "Policy_ID": rel.index,
            "Scenario": scen,
            "Relevance": rel[f"Relevance_{scen}"].values,
            "Group": grp[f"Group_{scen}"].values,
        })
        relevance_rows.append(rel_copy)

    if (it + 1) % 100 == 0:
        elapsed = time.time() - start_time
        print(f"  {it + 1}/{N_ITERATIONS} done ({elapsed:.1f}s elapsed)")

elapsed_total = time.time() - start_time
print(f"\n Monte Carlo complete: {N_ITERATIONS} iterations in {elapsed_total:.1f}s "
      f"({elapsed_total / N_ITERATIONS * 1000:.1f}ms/iteration)")

mc_topsis_p1_full = pd.concat(topsis_p1_rows, ignore_index=True)[["Iteration", "Policy_ID", "Rank", "TOPSIS Score"]]
mc_topsis_p2_full = pd.concat(topsis_p2_rows, ignore_index=True)[["Iteration", "Policy_ID", "Rank", "TOPSIS Score"]]
mc_relevance_full = pd.concat(relevance_rows, ignore_index=True)

mc_topsis_p1_full.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_TOPSIS_2030_2040.csv", index=False)    # <-- change this to your folder
mc_topsis_p2_full.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_TOPSIS_2040_2050.csv", index=False)     # <-- change this to your folder
mc_relevance_full.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_Relevance.csv", index=False)     # <-- change this to your folder

print(f"\n Saved: MC_TOPSIS_2030_2040.csv ({len(mc_topsis_p1_full)} rows)")
print(f" Saved: MC_TOPSIS_2040_2050.csv ({len(mc_topsis_p2_full)} rows)")
print(f" Saved: MC_Relevance.csv ({len(mc_relevance_full)} rows)")

## Stability check: how much does the TOP-ranked policy vary
print("\n--- Stability check: how often is each policy ranked #1 (Period 1)? ---")
print(mc_topsis_p1_full[mc_topsis_p1_full["Rank"] == 1]["Policy_ID"].value_counts().head(10))

PV_Roof exception applies to policies: ['I_16', 'I_17', 'I_18', 'I_19']

--- test run: 3 quick iterations to check for errors before scaling up ---
Iteration 0: Period1 top policy = N_07, Period2 top policy = N_07, Relevance_MIX range = [-0.458, 0.538]
Iteration 1: Period1 top policy = N_07, Period2 top policy = N_07, Relevance_MIX range = [-0.458, 0.538]
Iteration 2: Period1 top policy = N_07, Period2 top policy = N_07, Relevance_MIX range = [-0.458, 0.538]
 Smoke test passed, no errors

--- Running full Monte Carlo: 1000 iterations ---
  100/1000 done (2.3s elapsed)
  200/1000 done (4.1s elapsed)
  300/1000 done (5.8s elapsed)
  400/1000 done (7.5s elapsed)
  500/1000 done (9.2s elapsed)
  600/1000 done (11.1s elapsed)
  700/1000 done (12.8s elapsed)
  800/1000 done (14.5s elapsed)
  900/1000 done (16.2s elapsed)
  1000/1000 done (17.9s elapsed)

 Monte Carlo complete: 1000 iterations in 17.9s (17.9ms/iteration)

 Saved: MC_TOPSIS_2030_2040.csv (48000 rows)
 Saved: MC_TOPSIS_2040_205

## 5 Result visualization